# Chapter09
+ 아래 install 및 openai key입력을 진행해주세요
+ 파일이 colab 혹은 폴더 path에 넣어져 있는지 확인해주세요

In [ ]:
# 기존 설치된 주요 패키지 유지 + 9장 필수 패키지 추가
!pip install \
    langchain_openai==1.1.12 \
    langchain_community==0.4.1 \
    langchain==1.2.14 \
    sqlalchemy \
    numexpr \
    gradio \
    pydantic \
    tenacity \
    nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# 9장 필수 패키지 로드
import nest_asyncio
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent, create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.utilities import SQLDatabase
import warnings
warnings.filterwarnings("ignore")

nest_asyncio.apply()

import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

llm = ChatOpenAI(model="gpt-4o", temperature=0)

Enter your OpenAI API key: ··········


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_classic.tools import Tool, StructuredTool # Import StructuredTool
from pydantic import BaseModel, Field

# 1. 도구(Tools) 정의
def get_stock_history(symbol: str, period: str):
    # 실제 환경에서는 데이터베이스나 API에서 3개월치 데이터를 가져오는 로직이 들어갑니다.
    return f"{symbol}의 {period} 동안 주가는 96,500원에서 190,000원으로 96.9% 상승했습니다."
def get_financial_news(symbol: str):
    return f"{symbol} 관련 뉴스: 차세대 AI 메모리인 HBM4의 본격적인 양산 소식과 가격 인상 협상에 따른 수익성 개선 기대감 고조."

class StockHistoryInput(BaseModel):
    symbol: str = Field(description="주가 이력을 조회할 종목 코드 또는 이름")
    period: str = Field(description="주가 이력을 조회할 기간 (예: '최근 3개월', '1년')")

tools = [
    StructuredTool( # Use StructuredTool here
        name="get_stock_history",
        func=get_stock_history,
        description="특정 종목(symbol)의 주어진 기간(period, 예: '최근 3개월', '1년') 동안 주가 이력을 조회합니다.",
        args_schema=StockHistoryInput
    ),
    Tool(name="get_financial_news", func=get_financial_news, description="특정 종목의 최신 금융 뉴스를 가져옵니다.")
]

# 2. 메모리가 포함된 프롬프트 설계
# MessagesPlaceholder는 이전 대화 내용(chat_history)이 들어갈 자리를 비워둡니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 금융 데이터를 분석하는 전문 에이전트야. 도구를 사용하여 정확한 정보를 제공해."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"), # 에이전트의 중간 사고 과정이 기록되는 곳
])

# 3. 모델 및 에이전트 설정
llm = ChatOpenAI(model="gpt-4o", temperature=0)
agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 4. 세션별 메모리 관리 (멀티턴의 핵심)
demo_ephemeral_chat_history_for_chain = ChatMessageHistory()

conversational_agent_executor = RunnableWithMessageHistory(
    agent_executor,
    lambda session_id: demo_ephemeral_chat_history_for_chain,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 5. 실행 (연쇄 사고 테스트)
response = conversational_agent_executor.invoke(
    {"input": "최근 3개월 동안 삼성전자 주가 변동률과 관련 최신 뉴스 요약을 알려줘"},
    config={"configurable": {"session_id": "ch09_test_01"}}
)

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `get_stock_history` with `{'symbol': '삼성전자', 'period': '최근 3개월'}`


삼성전자의 최근 3개월 동안 주가는 96,500원에서 190,000원으로 96.9% 상승했습니다.
Invoking: `get_financial_news` with `삼성전자`


삼성전자 관련 뉴스: 차세대 AI 메모리인 HBM4의 본격적인 양산 소식과 가격 인상 협상에 따른 수익성 개선 기대감 고조.최근 3개월 동안 삼성전자의 주가는 96,500원에서 190,000원으로 약 96.9% 상승했습니다. 관련 최신 뉴스로는 삼성전자가 차세대 AI 메모리인 HBM4의 본격적인 양산을 시작했으며, 가격 인상 협상에 따른 수익성 개선 기대감이 고조되고 있다는 소식이 있습니다.

> Finished chain.
최근 3개월 동안 삼성전자의 주가는 96,500원에서 190,000원으로 약 96.9% 상승했습니다. 관련 최신 뉴스로는 삼성전자가 차세대 AI 메모리인 HBM4의 본격적인 양산을 시작했으며, 가격 인상 협상에 따른 수익성 개선 기대감이 고조되고 있다는 소식이 있습니다.


In [ ]:
from pydantic import BaseModel, Field, field_validator

class StockQuerySchema(BaseModel):
    symbol: str = Field(..., description="종목 코드")

    @field_validator("symbol")
    @classmethod
    def validate_symbol(cls, v: str):
        if not v.isupper():
            raise ValueError("종목 코드는 반드시 대문자여야 합니다.")
        return v


In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(
    stop=stop_after_attempt(3),  # 최대 3번까지 재시도
    wait=wait_exponential(multiplier=1, min=4, max=10), # 재시도 간격을 지수 함수적으로 증가
    reraise=True
)
def call_external_api(query):
    # 실제 외부 API 호출 로직
    response = requests.get(f"https://api.finance.com/v1/{query}")
    response.raise_for_status()
    return response.json()


In [ ]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_openai import ChatOpenAI
from langchain_classic.tools import Tool
from langchain_core.prompts import PromptTemplate

# 1. 도구(Tools) 정의
def get_stock_price(symbol: str) -> str:
    # 외부 API를 호출하거나 데이터베이스에서 조회하는 로직을 가정
    return f"{symbol} 현재 주가는 190,000원입니다."

# LangChain의 Tool은 기본적으로 인자를 하나 넘기려 시도하므로, 에러 방지용 가변 인자 추가
def get_exchange_rate(*args, **kwargs) -> str:
    return "현재 원달러 환율은 1,450원입니다."

tools = [
    Tool(
        name="get_stock_price",
        func=get_stock_price,
        description="특정 기업의 실시간 주가 정보를 조회합니다. 입력값은 기업 이름(예: 삼성전자)입니다."
    ),
    Tool(
        name="get_exchange_rate",
        func=get_exchange_rate,
        description="현재 원달러 환율 정보를 조회합니다."
    )
]

# 2. ReAct 전용 프롬프트 템플릿 설계 (10.2.2절 이론과 매칭)
template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

react_prompt = PromptTemplate.from_template(template)

# 3. 모델 및 에이전트 초기화
# ReAct 방식은 엄격한 텍스트 출력을 요구하므로 gpt-4o 모델 사용 권장
llm = ChatOpenAI(model="gpt-4o", temperature=0)

agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True # 모델이 형식을 어겼을 때 에러를 내지 않고 다시 시도하게 함
)

# 4. 에이전트 실행
response = agent_executor.invoke({"input": "삼성전자 주가와 현재 원달러 환율을 알려줘"})
print("\n[최종 답변]:", response["output"])



> Entering new AgentExecutor chain...
To answer the question, I need to find the current stock price of 삼성전자 and the current USD to KRW exchange rate.

Action: get_stock_price
Action Input: "삼성전자"삼성전자 현재 주가는 190,000원입니다.I have the current stock price of 삼성전자. Now, I need to find the current USD to KRW exchange rate.

Action: get_exchange_rate
Action Input: None현재 원달러 환율은 1,450원입니다.I now know the final answer.

Final Answer: 삼성전자 현재 주가는 190,000원이며, 현재 원달러 환율은 1,450원입니다.

> Finished chain.

[최종 답변]: 삼성전자 현재 주가는 190,000원이며, 현재 원달러 환율은 1,450원입니다.


In [ ]:
!pip install -U duckduckgo_search==7.5.1 yfinance ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.3 MB/s eta 0:00:00
  Attempting uninstall: curl_cffi
    Found existing installation: curl_cffi 0.14.0
    Uninstalling curl_cffi-0.14.0:
      Successfully uninstalled curl_cffi-0.14.0
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Successfully uninstalled yfinance-0.2.66


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("현대자동차 최근 뉴스")

'HyundaiShop.현대자동차공식 온라인몰에서 다양한 상품을 확인해보세요.현대자동차소식. All-in-One 구매 가이드. 현대자동차現代自動車｜HYUNDAI MOTOR.정식:현대자동차주식회사 한문: 現代自動車 株式會社 영문: Hyundai Motor Company. 국가. 이 유튜버는최근현대자동차모 직영점에 방문해 차량을 구경했다. 전화로 먼저 예약을 해뒀다는 그는 도착해서 영상 촬영이 가능하냐고 문의했다. 데이터. 기업.뉴스.뉴스리스트. 기업의 주요 활동과 관련된 뉴스가 표기됩니다. О сервисе Прессе Авторские права Связаться с нами Авторам Рекламодателям...'

In [ ]:
import yfinance as yf
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_openai import ChatOpenAI
from langchain_classic.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun

# 1. 실제 금융 데이터 API 도구 (Yahoo Finance)
def get_financial_statement(symbol: str) -> str:
    """특정 기업의 티커를 입력받아 시가총액과 PER 등 재무 정보를 반환합니다."""
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info
        market_cap = info.get('marketCap', '정보 없음')
        forward_pe = info.get('forwardPE', '정보 없음')

        # 시가총액을 조 단위로 변환 (숫자일 경우)
        if isinstance(market_cap, (int, float)):
            market_cap = f"{market_cap / 1_000_000_000_000:.2f}조 원"

        return f"{symbol}의 시가총액은 {market_cap}이며, 추정 PER은 {forward_pe}입니다."
    except Exception as e:
        return f"재무 정보를 불러오는 데 실패했습니다: {e}"

# 2. 실시간 뉴스 검색 도구 (DuckDuckGo)
search_tool = DuckDuckGoSearchRun()

# 3. 텍스트 요약 도구 (LLM을 도구 안에서 다시 호출)
def summarize_text(text: str) -> str:
    """검색된 긴 텍스트를 입력받아 핵심만 3줄로 요약합니다."""
    summary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = summary_llm.invoke(f"다음 뉴스 텍스트를 3문장 이내로 핵심만 요약해:\n{text}")
    return response.content

# 4. 도구 리스트 및 정교한 설명(Description) 설정
tools = [
    Tool(
        name="get_financial_statement",
        func=get_financial_statement,
        description="기업의 재무 정보(시가총액, PER 등)를 조회합니다. 한국 주식은 종목코드 뒤에 '.KS'를 붙이세요 (예: 현대차 -> 005380.KS)."
    ),
    Tool(
        name="get_latest_news",
        func=search_tool.run,
        description="기업 관련 최신 뉴스를 검색합니다."
    ),
    Tool(
        name="summarize_text",
        func=summarize_text,
        description="매우 긴 뉴스나 검색 결과를 요약할 때 사용합니다."
    )
]

# 5. ReAct 표준 프롬프트
template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer in KOREAN to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

react_prompt = PromptTemplate.from_template(template)

# 6. 모델 및 에이전트 실행기 구성
llm = ChatOpenAI(model="gpt-4o", temperature=0)

agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# 7. 실행
user_query = "한국 주식 현대차(005380.KS)의 실제 재무 정보와, 최근 현대자동차 관련 뉴스를 검색해서 한국어로 요약해줘."
response = agent_executor.invoke({"input": user_query})

print("\n[최종 답변]:\n", response["output"])



> Entering new AgentExecutor chain...
현대차의 재무 정보를 먼저 조회한 후, 최신 뉴스를 검색하여 요약하겠습니다.

Action: get_financial_statement
Action Input: "005380.KS"005380.KS의 시가총액은 123.12조 원이며, 추정 PER은 9.1299715입니다.현대차의 재무 정보를 확인했습니다. 이제 현대차 관련 최신 뉴스를 검색하여 요약하겠습니다.

Action: get_latest_news
Action Input: "현대자동차"HyundaiShop.현대자동차공식 온라인몰에서 다양한 상품을 확인해보세요.현대자동차소식. All-in-One 구매 가이드. 현대자동차공식 유튜브 채널 Hyundai Motor Company Official YouTube Channel.현대자동차(hyundai korea). 2,4 млн просмотров 9 дней назад. 배려가 느껴지는 디자인의 #현대자동차신형 #아반떼 #Hyundai_Motor #Avante has external #design with refined dynamic and considerate design. | 자동차, 외관 디자인. 10.1M개의 게시물이 있습니다. TikTok에서현대자동차생산직 초탁 관련 동영상을 찾아보세요. @hyundai.min현대자동차민성준. 한눈에 보는 자동차 사용 설명서.Thought: 최신 뉴스 검색 결과는 현대자동차의 다양한 온라인 활동과 신형 아반떼의 디자인에 대한 내용이 포함되어 있습니다. 이를 요약하겠습니다.

Final Answer: 현대차의 시가총액은 123.12조 원이며, 추정 PER은 9.13입니다. 최근 현대자동차 관련 뉴스에서는 현대자동차의 공식 온라인몰과 유튜브 채널에서 다양한 상품과 콘텐츠를 제공하고 있으며, 신형 아반떼의 세련되고 배려 깊은 외관 디자인이 주목받고 있습니다. TikTok에서는 현대자동차 생산직 관련 동영상도 인기를 끌고 있습니다.

> Finished c

In [ ]:
import requests
import warnings
warnings.filterwarnings("ignore")

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# 1. 실제 날씨 API 도구 정의 (Open-Meteo 사용)
@tool
def get_weather(location: str) -> str:
    """특정 지역의 현재 날씨와 기온을 조회합니다.
    location: 조회할 도시 이름. (주의: API 검색을 위해 '서울'은 'Seoul'처럼 반드시 영문으로 번역해서 입력하세요)
    """
    try:
        # Step 1: 도시 이름을 위도(Latitude)와 경도(Longitude)로 변환 (Geocoding API)
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=ko"
        geo_resp = requests.get(geo_url).json()

        if "results" not in geo_resp:
            return f"{location}의 좌표를 찾을 수 없습니다. 도시 이름을 영어로 다시 시도해보세요."

        lat = geo_resp["results"][0]["latitude"]
        lon = geo_resp["results"][0]["longitude"]

        # Step 2: 위경도를 바탕으로 현재 기온 조회 (Weather Forecast API)
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_resp = requests.get(weather_url).json()

        current = weather_resp.get("current_weather", {})
        temp = current.get("temperature", "알 수 없음")
        windspeed = current.get("windspeed", "알 수 없음")

        return f"{location}의 현재 기온은 {temp}도이며, 풍속은 {windspeed}km/h입니다."

    except Exception as e:
        return f"날씨 정보를 불러오는 중 에러가 발생했습니다: {e}"

# 2. 모델 초기화 및 도구 바인딩
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# bind_tools를 통해 모델에게 "필요하면 이 도구를 써도 좋아"라고 JSON 스키마를 전달합니다.
llm_with_tools = llm.bind_tools([get_weather])

# 3. 모델 호출
user_query = "지금 서울 날씨 어때? 덥진 않아?"
response = llm_with_tools.invoke(user_query)

# 4. Function Calling 결과 파싱 및 실행
# 모델이 일반 텍스트 대답 대신 함수 호출(tool_calls)을 제안했는지 확인합니다.
if response.tool_calls:
    tool_call = response.tool_calls[0]
    name = tool_call["name"]
    args = tool_call["args"]

    print(f"[LLM의 판단 (Function Calling)]")
    print(f"호출할 함수: {name}")
    print(f"추출된 인자: {args}\n")

    # 5. 애플리케이션 단에서 실제 파이썬 함수를 실행하고 결과 출력
    result = get_weather.invoke(args)
    print(f"[실측 데이터 결과]\n{result}")
else:
    print(response.content)

[LLM의 판단 (Function Calling)]
호출할 함수: get_weather
추출된 인자: {'location': 'Seoul'}

[실측 데이터 결과]
Seoul의 현재 기온은 4.4도이며, 풍속은 4.1km/h입니다.


In [ ]:
import os
import yfinance as yf
import warnings
warnings.filterwarnings("ignore") # 불필요한 경고 메시지 숨김

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# 1. 실제 금융 데이터 API 도구 (Yahoo Finance 연동)

@tool
def get_stock_price(symbol: str) -> str:
    """특정 종목의 실시간 주가 정보를 조회합니다.
    주의: 한국 주식은 반드시 종목코드 뒤에 '.KS'를 붙여야 합니다 (예: 삼성전자 -> 005930.KS).
    """
    try:
        ticker = yf.Ticker(symbol)
        # fast_info를 사용하면 가볍고 빠르게 현재가를 가져옵니다.
        price = ticker.fast_info['last_price']
        return f"{symbol}의 현재 주가는 {price:,.0f}원입니다."
    except Exception as e:
        return f"주가 정보를 가져오는 데 실패했습니다. 심볼을 확인해주세요: {symbol}"

@tool
def get_exchange_rate(currency_pair: str = "USDKRW=X") -> str:
    """환율 정보를 조회합니다. 기본값은 USD/KRW 환율을 의미하는 'USDKRW=X'입니다."""
    try:
        ticker = yf.Ticker(currency_pair)
        rate = ticker.fast_info['last_price']
        return f"현재 {currency_pair} 환율은 {rate:,.2f}원입니다."
    except Exception as e:
        return f"환율 정보를 가져오는 데 실패했습니다: {currency_pair}"

@tool
def summarize_financials(symbol: str) -> str:
    """특정 기업의 재무 요약 정보(매출, 영업이익률 등)를 제공합니다.
    주의: 한국 주식은 종목코드 뒤에 '.KS'를 붙여야 합니다 (예: 삼성전자 -> 005930.KS).
    """
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info

        revenue = info.get('totalRevenue', 0)
        margins = info.get('operatingMargins', 0) * 100

        # 매출을 보기 쉽게 조 단위로 변환
        if revenue > 0:
            revenue_str = f"{revenue / 1_000_000_000_000:.2f}조 원"
        else:
            revenue_str = "정보 없음"

        return f"{symbol}의 최근 매출은 {revenue_str}이며, 영업이익률은 약 {margins:.2f}%입니다."
    except Exception as e:
        return f"재무 정보를 불러오는 데 실패했습니다: {symbol}"

# 2. 모델 초기화 및 도구 바인딩
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [get_stock_price, get_exchange_rate, summarize_financials]

# bind_tools를 통해 모델에게 도구의 명세서(Schema)를 전달합니다.
llm_with_tools = llm.bind_tools(tools)

# 3. 에이전트 실행 로직
user_input = "삼성전자 주가와 현재 환율, 그리고 삼성전자의 재무 상태를 요약해줘."
ai_msg = llm_with_tools.invoke(user_input)

# 4. 도구 호출 제안 확인 및 실제 실행
if ai_msg.tool_calls:
    print("=== [LLM의 도구 호출 제안 및 실행 결과] ===\n")
    for tool_call in ai_msg.tool_calls:
        # tool_call["name"]과 실제 함수 객체를 매핑
        selected_tool = {
            "get_stock_price": get_stock_price,
            "get_exchange_rate": get_exchange_rate,
            "summarize_financials": summarize_financials
        }[tool_call["name"].lower()]

        # 도구 실제 실행
        tool_output = selected_tool.invoke(tool_call["args"])

        print(f"🛠️ [실행 도구]: {tool_call['name']}")
        print(f"📥 [입력 파라미터]: {tool_call['args']}")
        print(f"📤 [실측 결과]: {tool_output}\n")
else:
    print(ai_msg.content)

=== [LLM의 도구 호출 제안 및 실행 결과] ===

🛠️ [실행 도구]: get_stock_price
📥 [입력 파라미터]: {'symbol': '005930.KS'}
📤 [실측 결과]: 005930.KS의 현재 주가는 196,500원입니다.

🛠️ [실행 도구]: get_exchange_rate
📥 [입력 파라미터]: {}
📤 [실측 결과]: 현재 USDKRW=X 환율은 1,501.89원입니다.

🛠️ [실행 도구]: summarize_financials
📥 [입력 파라미터]: {'symbol': '005930.KS'}
📤 [실측 결과]: 005930.KS의 최근 매출은 333.61조 원이며, 영업이익률은 약 21.32%입니다.



## 10.4.2

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
import sqlite3

# 1. 파일 기반 SQLite DB 생성 및 샘플 데이터 주입
db_path = "hospital.db"

# 코랩에서 셀을 여러 번 실행할 때 에러가 나지 않도록 기존 파일 삭제
if os.path.exists(db_path):
    os.remove(db_path)

# ':memory:' 대신 실제 물리적 파일로 DB 생성
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.executescript("""
CREATE TABLE doctors (
    id INTEGER PRIMARY KEY,
    name TEXT,
    specialty TEXT,
    email TEXT
);
CREATE TABLE appointments (
    app_id INTEGER PRIMARY KEY,
    doctor_id INTEGER,
    patient_name TEXT,
    available_time TEXT,
    FOREIGN KEY(doctor_id) REFERENCES doctors(id)
);
INSERT INTO doctors VALUES (1, '김재준', '내과', 'kim_int@hospital.com');
INSERT INTO appointments VALUES (101, 1, '이나라', '2025-05-10 14:00');
""")
conn.commit()

db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ==========================================
# 2. [핵심] SQL 에이전트에 멀티턴(기억력) 프롬프트 장착
# ==========================================
# 랭체인 기본 SQL 지시문을 가져옵니다.
system_message = SQL_PREFIX.format(dialect="SQLite", top_k=5)

# 10.1.1절에서 배운 'chat_history' 예약석을 프롬프트에 추가합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    MessagesPlaceholder(variable_name="chat_history"), # 이전 대화 기억 공간
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# 3. 에이전트 생성 (커스텀 프롬프트 적용)
agent_executor = create_sql_agent(
    llm=llm,
    db=db,
    prompt=prompt, # 기억력이 추가된 프롬프트 주입
    agent_type="openai-tools",
    verbose=True
)

# 4. 메모리 래퍼 적용 (10.1.1절과 완벽히 동일한 원리)
chat_history = ChatMessageHistory()
conversational_agent = RunnableWithMessageHistory(
    agent_executor,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# ==========================================
# 5. 실행 및 테스트
# ==========================================
print("=== 1차 질문 ===")
response = conversational_agent.invoke(
    {"input": "김재준 의사의 이메일이 뭐야?"},
    config={"configurable": {"session_id": "hospital_session"}} # 세션 ID로 대화방 구분
)
print(f"답변: {response['output']}\n")

print("=== 2차 질문 ===")
response_2 = conversational_agent.invoke(
    {"input": "그 의사의 예약 가능 시간은 언제야?"},
    config={"configurable": {"session_id": "hospital_session"}}
)
print(f"답변: {response_2['output']}")

=== 1차 질문 ===


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


appointments, doctors
Invoking: `sql_db_schema` with `{'table_names': 'doctors'}`



CREATE TABLE doctors (
	id INTEGER, 
	name TEXT, 
	specialty TEXT, 
	email TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from doctors table:
id	name	specialty	email
1	김재준	내과	kim_int@hospital.com
*/김재준 의사의 이메일은 "kim_int@hospital.com"입니다.

> Finished chain.
답변: 김재준 의사의 이메일은 "kim_int@hospital.com"입니다.

=== 2차 질문 ===


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


appointments, doctors
Invoking: `sql_db_schema` with `{'table_names': 'appointments, doctors'}`



CREATE TABLE appointments (
	app_id INTEGER, 
	doctor_id INTEGER, 
	patient_name TEXT, 
	available_time TEXT, 
	PRIMARY KEY (app_id), 
	FOREIGN KEY(doctor_id) REFERENCES doctors (id)
)

/*
3 rows from appointments table:
app_id	doctor_id	patient_name	available_time
101	1	이나라	2025-05-10 14:00
*/


CREATE TABLE doctor